<a href="https://colab.research.google.com/github/Abhi01shinde/Python-project/blob/main/Infotact_Final_Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project 2:
FinTech (NLP) - Intelligent Document Parsing (NER)(NAMED ENTITY RECOGNITION)

Project Title: **Automated Legal Entity Extractor**
Product Brand Name: **"LexiScan Auto"**




**Week 1**


---

Data
preparation for
training. Tasks: Build
the robust OCR
pipeline. Annotate a
representative, small
subset of contracts
manually using a tool
like Doccano or
Prodigy (labeling
tags like B-PARTY,
I-PARTY, BAMOUNT).

In [ ]:
# Install system packages
!apt-get install -y tesseract-ocr
!apt-get install -y poppler-utils

# Install Python libraries
!pip install pytesseract pdf2image opencv-python pillow

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 5 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (325 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processin

In [ ]:
from google.colab import files

uploaded = files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import os
from pdf2image import convert_from_path
import pytesseract
import cv2

# Create output folder
output_folder = "extracted_texts"
os.makedirs(output_folder, exist_ok=True)

for file_name in uploaded.keys():

    if file_name.lower().endswith(".pdf"):
        print(f"\nProcessing: {file_name}")

        # Convert PDF to images
        pages = convert_from_path(file_name)

        full_text = ""

        for i, page in enumerate(pages):
            image_path = f"temp_page_{i}.png"
            page.save(image_path, "PNG")

            # Read image
            img = cv2.imread(image_path)

            # Convert to grayscale
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # Improve OCR accuracy
            gray = cv2.medianBlur(gray, 3)
            _, gray = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

            # Extract text
            text = pytesseract.image_to_string(gray)

            full_text += text + "\n"

            os.remove(image_path)

        # Save as .txt
        txt_name = file_name.replace(".pdf", ".txt")
        txt_path = os.path.join(output_folder, txt_name)

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(full_text)

        print(f"Saved: {txt_path}")

In [ ]:
from google.colab import files
import os

folder_path = "extracted_texts"

for file_name in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file_name)
    print("Downloading:", file_name)
    files.download(file_path)

**WEEK 2**

---
Contextual
understanding of text.
Tasks: Train a
custom Spacy NER
model or a deep
learning Bi-LSTM
sequence model in
TensorFlow using the
annotated data.
We have choosen Spacy NER



In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

In [ ]:
import spacy
from spacy.training.example import Example
import json

# Load blank English model
nlp = spacy.blank("en")

# Add NER pipeline if not present
if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")

# Add labels from your dataset
labels = ["DATE", "PARTY", "MONEY"]
for label in labels:
    ner.add_label(label)

# Load JSONL data
train_data = []
with open("/content/admin.jsonl", "r") as f:
    for line in f:
        doc_json = json.loads(line)
        text = doc_json["text"]
        entities = [(start, end, label) for start, end, label in doc_json["label"]]
        train_data.append((text, {"entities": entities}))

print(f"Loaded {len(train_data)} training examples.")

In [ ]:
import random
from spacy.util import minibatch, compounding

# Training settings
optimizer = nlp.begin_training()
n_iter = 50  # Increase for better results

for i in range(n_iter):
    random.shuffle(train_data)
    losses = {}
    batches = minibatch(train_data, size=compounding(2.0, 16.0, 1.5))
    for batch in batches:
        texts, annotations = zip(*batch)
        for text, annots in zip(texts, annotations):
            doc = nlp.make_doc(text)
            example = Example.from_dict(doc, annots)
            nlp.update([example], drop=0.3, losses=losses)
    print(f"Iteration {i+1}, Losses: {losses}")

# Save the trained model
nlp.to_disk("contract_ner_model")

In [ ]:
# Load trained model
nlp2 = spacy.load("contract_ner_model")

# Test
test_text = "Agreement dated 15 April 2024 between Beta Corp and Gamma LLC for $55,000."
doc = nlp2(test_text)
for ent in doc.ents:
    print(ent.text, ent.label_)

WEEK-3 RULE BASED LAYER & PRECISION

MAXIMIZING PRECISION AND HANDLING EDGE CASES.

Maximizing Precision : Precision measures how many of the entities the model identified are actually correct.To maximize precision,for Refine Training data, Feature Engineering, Error Analysis, Rule-Based Adjustments.

Handling Edge Cases: Edge cases are unusual or less commom linguistic patterns that the model might not hace encountered sufficiently during training.These can lead to misclassifications or missed entities. To handle them: Expand Training Data with Diverse, Contextual rules, Poast-Processing, Iterative Improvement.

In [ ]:
import re
from datetime import datetime

# Example NER output
entities = [
    ("March 5, 2024", "DATE"),
    ("05/03/2024", "DATE"),
    ("2024-03-05", "DATE"),
    ("$5,000", "MONEY"),
    ("$3,O00", "MONEY")  # OCR error
]

cleaned_entities = []


# -------- DATE STANDARDIZATION --------
def standardize_date(date_text):

    formats = [
        "%B %d, %Y",
        "%d/%m/%Y",
        "%Y-%m-%d",
        "%d-%m-%Y"
    ]

    for fmt in formats:
        try:
            date_obj = datetime.strptime(date_text, fmt)
            return date_obj.strftime("%Y-%m-%d")  # ISO 8601
        except:
            continue

    return None


# -------- MONEY CLEANUP --------
def clean_money(money_text):

    # Fix OCR errors
    money_text = money_text.replace("O", "0")

    # Remove symbols and commas
    value = re.sub(r"[^\d]", "", money_text)

    if value:
        return int(value)

    return None


# -------- RULE BASED CLEANUP --------
for text, label in entities:

    if label == "DATE":
        normalized = standardize_date(text)

        if normalized:   # constraint: valid date
            cleaned_entities.append((normalized, label))

    elif label == "MONEY":
        value = clean_money(text)

        if value:        # constraint: must contain numbers
            cleaned_entities.append((value, label))

print(cleaned_entities)

[('2024-03-05', 'DATE'), ('2024-03-05', 'DATE'), ('2024-03-05', 'DATE'), (5000, 'MONEY'), (3000, 'MONEY')]


In [ ]:
def test_date_format():
    assert standardize_date("March 5, 2024") == "2024-03-05"


def test_date_slash():
    assert standardize_date("05/03/2024") == "2024-03-05"


def test_money_clean():
    assert clean_money("$5,000") == 5000


def test_ocr_error():
    assert clean_money("$3,O00") == 3000

TASKS: IMPLEMENT HEURISTIC AND CONSTRAINT-BASED RULES

In [ ]:
import re
from dateutil import parser
from datetime import datetime

def standardize_date(date_text):

    try:
        # remove ordinal suffixes (1st, 2nd, 3rd, 4th...)
        date_text = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', date_text)

        # parse date using dateutil
        parsed_date = parser.parse(date_text)

        # constraint check
        if parsed_date.year < 1900 or parsed_date.year > 2100:
            return "Invalid Year"

        # convert to ISO 8601
        return parsed_date.strftime("%Y-%m-%d")

    except Exception:
        return "Invalid Date"

TESTNG WITH MULTIPLE FORMATS

In [ ]:
dates = [
    "12/05/2023",
    "May 12th, 2023",
    "2023.05.12",
    "5th May 23",
    "MAY 12 2023"
]

WEEK-4: CONTAINERIZATION & DELIVERABLE

TASKS: Dockerize the entire application.

Dockerizing the entire application means packaging the application code, NLP model, OCR tools, and all dependencies into one container so it can run the same way on any machine.

TASKS: OCR Function(Extract text from PDF)

In [ ]:
import spacy

# Load pre-trained model
nlp = spacy.load("en_core_web_sm")

def extract_entities(text):

    doc = nlp(text)

    entities = []

    for ent in doc.ents:
        entities.append({
            "text": ent.text,
            "label": ent.label_
        })

    return entities

In [ ]:
{
 "PARTY": "ABC Corporation",
 "DATE": "2023-05-12",
 "AMOUNT": "$50000",
 "JURISDICTION": "California"
}

{'PARTY': 'ABC Corporation',
 'DATE': '2023-05-12',
 'AMOUNT': '$50000',
 'JURISDICTION': 'California'}

RESULTING API AND STRUCTURED JSON OBJECT:

In [ ]:
from fastapi import FastAPI, UploadFile
import shutil
from utils.ocr import extract_text_from_pdf
from utils.extractor import extract_entities

app = FastAPI()

@app.post("/extract")
async def extract_entities_api(file: UploadFile):

    file_path = "temp.pdf"

    with open(file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    text = extract_text_from_pdf(file_path)

    entities = extract_entities(text)

    return {"entities": entities}

In [ ]:
{
 "entities": {
   "PARTY": "ABC Corporation",
   "DATE": "2023-05-12",
   "AMOUNT": "$50000",
   "JURISDICTION": "California"
 }
}

{'entities': {'PARTY': 'ABC Corporation',
  'DATE': '2023-05-12',
  'AMOUNT': '$50000',
  'JURISDICTION': 'California'}}

END-To-END TEST:

In [ ]:
from ocr import extract_text_from_pdf
from ner import extract_entities
import json


def run_pipeline(pdf_path):

    print("Step 1: Running OCR...")

    text = extract_text_from_pdf(pdf_path)

    print("Step 2: Extracting entities...")

    entities = extract_entities(text)

    result = {
        "document": pdf_path,
        "entities": entities
    }

    return result


if __name__ == "__main__":

    pdf_file = "heldout_contract.pdf"

    output = run_pipeline(pdf_file)

    print("\nFinal Output:\n")

    print(json.dumps(output, indent=4))

In [ ]:
{
 "document": "heldout_contract.pdf",
 "entities": [
   {
     "text": "ABC Corporation",
     "label": "ORG"
   },
   {
     "text": "May 12, 2023",
     "label": "DATE"
   },
   {
     "text": "$50,000",
     "label": "MONEY"
   },
   {
     "text": "California",
     "label": "GPE"
   }
 ]
}

{'document': 'heldout_contract.pdf',
 'entities': [{'text': 'ABC Corporation', 'label': 'ORG'},
  {'text': 'May 12, 2023', 'label': 'DATE'},
  {'text': '$50,000', 'label': 'MONEY'},
  {'text': 'California', 'label': 'GPE'}]}